# Text Mining Project - Final Model
**Spring Semester 2025/2026**

Group 11:
- Ana Macedo
- Carlota Pires
- Francisca Calçoa
- Francisca Martins



<hr>
<a class="anchor" id="one-bullet">
<d style="color:white;">

# 1. Imports and Load Data
</a>
</d>   

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train = pd.read_csv('/content/drive/MyDrive/train.csv')
test = pd.read_csv('/content/drive/MyDrive/test.csv')

X_train_raw = train['text'].astype(str)
y_train_raw = train['label']

X_test_raw = test['text'].astype(str)

In [ ]:
train = pd.read_csv('../Datasets/train.csv')
test = pd.read_csv('../Datasets/test.csv')

X_train_raw = train['text'].astype(str)
y_train_raw = train['label']

X_test_raw = test['text'].astype(str)

<hr>
<a class="anchor" id="two-bullet">
<d style="color:white;">

# 2. Final Model Pipeline
</a>
</d>   
!! here we must put the final model as a pipeline!!!

depois de ter o modelo ajustar isto

In [5]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=train['label'].nunique()
)

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(X_train_raw, y_train_raw)
test_dataset = TweetDataset(X_test_raw)

training_args = TrainingArguments(
    output_dir="./bertweet-final",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3582, training_loss=0.20941632822996276, metrics={'train_runtime': 259.535, 'train_samples_per_second': 220.618, 'train_steps_per_second': 13.802, 'total_flos': 2971875303318852.0, 'train_loss': 0.20941632822996276, 'epoch': 6.0})

### Export Test Predictions

In [7]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=1)

submission = pd.DataFrame({
    "id": test["id"],
    "label": pred_labels
})

submission.to_csv("pred_11.csv", index=False)

In [8]:
from google.colab import files
files.download("pred_11.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>